# Compute the HelpSteer2 Relationship Matrix

This notebook computes a static relationship matrix $R$ from five local GPT-2 LoRA adapters trained for the HelpSteer2 attributes **helpfulness**, **correctness**, **coherence**, **complexity**, and **verbosity**.

The thesis pipeline is:

$$\delta_i \rightarrow R \rightarrow \lambda = f(p, R) \rightarrow \theta(\lambda)$$

This notebook performs only the $\delta_i \rightarrow R$ step. It does not train adapters, evaluate generated responses, merge models, or run the M1 preference-to-coefficient mapping.

## Clone or update the repository

The following cell starts from `/content`. It updates an existing valid repository or clones a fresh copy, avoiding nested folders such as `/content/master-thesis/master-thesis`.

In [ ]:
from pathlib import Path
import os
import subprocess

repo_path = Path("/content/master-thesis")
repo_url = "https://github.com/NZhang137/master-thesis.git"

os.chdir("/content")

if (repo_path / ".git").is_dir():
    print("Repository found. Pulling the latest changes...")
    subprocess.run(["git", "-C", str(repo_path), "pull"], check=True)
elif repo_path.exists():
    raise RuntimeError(
        f"{repo_path} exists but is not a Git repository. "
        "Rename or remove it, then run this cell again."
    )
else:
    print("Cloning the repository...")
    subprocess.run(["git", "clone", repo_url, str(repo_path)], check=True)

os.chdir(repo_path)
print(f"Current folder: {Path.cwd()}")

## Show the repository structure

Confirm that the repository contains the relationship-matrix script and source utilities.

In [ ]:
!pwd
!ls
!ls scripts
!ls src

## Install dependencies

The computation uses PyTorch for tensor operations, `safetensors` for loading PEFT adapter weights, and pandas for the matrix preview. Colab normally includes PyTorch, but the command ensures that all required packages are available.

The script reads saved PEFT files directly, so the `peft` package itself is not required for this step.

In [ ]:
!pip install -q torch pandas safetensors

In [ ]:
import torch
import pandas as pd
import safetensors

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("pandas version:", pd.__version__)
print("safetensors version:", safetensors.__version__)

## Check whether the HelpSteer2 adapters exist

All five local adapter folders are required. If every folder is marked `FOUND`, skip the upload section.

In [ ]:
from pathlib import Path

adapter_paths = [
    Path("adapters/helpsteer2-gpt2-helpfulness-adapter"),
    Path("adapters/helpsteer2-gpt2-correctness-adapter"),
    Path("adapters/helpsteer2-gpt2-coherence-adapter"),
    Path("adapters/helpsteer2-gpt2-complexity-adapter"),
    Path("adapters/helpsteer2-gpt2-verbosity-adapter"),
]

for path in adapter_paths:
    status = "FOUND" if path.is_dir() else "MISSING"
    print(f"{status:7} {path}")

all_adapters_exist = all(path.is_dir() for path in adapter_paths)
print(f"\nAll adapters available: {all_adapters_exist}")

## Upload the adapter backup if needed

Run the next two cells only when one or more adapters are missing. Select your local `helpsteer2_adapters.zip` backup.

**Important:** The ZIP file is only a local backup of generated adapter weights. Neither the ZIP nor the extracted `adapters/` folder should be committed to GitHub.

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
!if [ -f helpsteer2_adapters.zip ]; then unzip -o helpsteer2_adapters.zip; else echo "No helpsteer2_adapters.zip found, skipping unzip."; fi
!ls adapters || echo "No adapters folder found."

## Check the adapter files

The checker confirms that all five folders contain the expected PEFT configuration and adapter-weight files. Continue only after it reports success.

In [ ]:
!python scripts/check_helpsteer2_adapters.py

## Compute the relationship matrix

The script loads only the saved LoRA tensors, flattens each adapter into a vector, and computes pairwise cosine similarities. It does not load a full GPT-2 model.

In [ ]:
!python scripts/compute_helpsteer2_relationship_matrix.py

## Inspect the output files

The script creates two small files:

- `results/helpsteer2_relationship_matrix.csv`
- `results/helpsteer2_relationship_matrix_metadata.json`

The CSV stores the labeled $5 \times 5$ matrix. The JSON records the adapter names and paths, representation, vector lengths, similarity type, and prototype caveat.

In [ ]:
!ls results
!cat results/helpsteer2_relationship_matrix.csv
!cat results/helpsteer2_relationship_matrix_metadata.json

In [ ]:
relationship_df = pd.read_csv(
    "results/helpsteer2_relationship_matrix.csv",
    index_col="adapter",
)

relationship_df

## Understanding the matrix

The rows and columns correspond to helpfulness, correctness, coherence, complexity, and verbosity. Each value is the cosine similarity between two flattened LoRA adapter parameter vectors.

Diagonal values should be close to `1.0`. Off-diagonal values describe geometric alignment between different adapters. This matrix is the $R$ input intended for the later $\lambda = f(p, R)$ stage, but it remains a prototype proxy for task-vector relationships and requires empirical validation.

## Git safety check

It is acceptable to commit the small CSV and JSON result files.

Do not commit:

- `adapters/`
- ZIP backup files
- `.safetensors` or `.bin` files
- checkpoints or full model files

In [ ]:
!git status

## What this notebook establishes

This notebook computes only the five-objective HelpSteer2 relationship matrix $R$ from LoRA adapter geometry. Preference-to-coefficient mapping remains a later step.